# SEPA — Exploración de Productos y Canasta Representativa

**SEPA:** Sistema Electrónico de Publicidad de Precios Argentinos — publica diariamente los precios reportados por las principales cadenas de supermercados del país.

**Objetivo:** Explorar los datos del SEPA de **abril 2026** para identificar qué productos tienen alta cobertura a nivel nacional (cadenas y provincias) y generar dos insumos:

1. **Canasta representativa sugerida** — selección automática de productos con máxima cobertura geográfica y comercial, organizada por grupos alimentarios y del hogar, para una familia tipo de 4 integrantes.
2. **Lista de candidatos** — todos los productos que superan los umbrales mínimos de cobertura, con sus métricas completas, para que un economista pueda armar o ajustar su propia canasta.

Ambos insumos se exportan a `canasta_representativa_{PERIODO}.xlsx` con las hojas **Canasta** y **Candidatos**. El Excel está pensado como **fuente de alimentación** de notebooks posteriores de análisis temporal, comparación con IPC y mapas geográficos.

**Fuentes de datos:**
- **Datos SEPA** (ZIPs semestrales): carpeta `carga/` en Google Drive (`2024A.zip`, `2024B.zip`, `2025A.zip`, `2025B.zip`, `2026A.zip`)
- **Maestros** (productos y sucursales): incluidos en el repositorio, con descarga automática desde GitHub

**Estructura del notebook:**
1. Configuración de rutas y parámetros
2. Carga de datos SEPA — Abril 2026
3. Carga de maestros
4. Enriquecimiento y exploración
5. Análisis de cobertura
6. Construcción de la canasta
7. Exportación de resultados

---
> **Nota sobre precios:** Los datos SEPA pueden estar en centavos (datos pre-2025) o en pesos (datos 2025B+). El notebook **autodetecta el factor de conversión** al cargar cada período y ajusta automáticamente.

## 1. Configuración

**Solo modificar esta sección.** Ajustar los paths según la ubicación de los archivos en tu equipo.

In [ ]:
# ===========================================================
# CONFIGURACIÓN — Solo modificar esta sección
# ===========================================================
#
# SEPA_SOURCE controla de dónde se toman los datos del SEPA:
#
#   'mi_drive' → tus propios ZIPs en Google Drive (Default)
#               La carpeta 'carga/' en tu Drive contiene los ZIPs semestrales:
#               2024A.zip, 2024B.zip, 2025A.zip, 2025B.zip, 2026A.zip
#               Ajustar SEPA_DIR si la carpeta tiene otro nombre o ubicación.
#
#   'local'    → ejecución fuera de Colab (Windows/Mac)
#               Ajustar SEPA_DIR con el path en tu equipo.

SEPA_SOURCE = 'mi_drive'

# ── Completar solo si SEPA_SOURCE es 'mi_drive' o 'local' ──
SEPA_DIR     = '/content/drive/MyDrive/carga'
MAESTROS_DIR = None
OUTPUT_DIR   = '/content/drive/MyDrive/carga/output_canasta'
# ────────────────────────────────────────────────────────────

# Período a analizar
# ─ None      → autodetectar el último mes disponible en SEPA_DIR
# ─ 'YYYY-MM' → forzar un mes específico (ej: '2026-04', '2025-12')
PERIODO = None

# Parámetros de cobertura
# MIN_CADENAS y MIN_PROVINCIAS se calculan dinámicamente (el producto debe estar
# presente en TODOS los grupos corporativos y TODAS las provincias activas)
MIN_SUCURSALES = 50    # reportado por al menos 50 sucursales
MIN_PCT_DIAS   = 0.50  # precio disponible en al menos el 50% de los días

# Umbrales para hoja Selección — más amplios para dar opciones al economista
# La Canasta automática sigue usando los umbrales estrictos de arriba.
MIN_CADENAS_SEL    = 3     # al menos 3 de 5 grupos corporativos
MIN_PROVINCIAS_SEL = 18    # al menos 18 de 24 provincias
MIN_SUCURSALES_SEL = 30    # al menos 30 sucursales

# Caché de parquets (evita reprocesar desde cero si el kernel de Colab crashea)
# El caché se guarda en OUTPUT_DIR/_cache/ y se invalida automáticamente si
# se cambia PERIODO.
USE_CACHE = True

In [ ]:
# Montar Google Drive (solo necesario para SEPA_SOURCE = 'mi_drive')
try:
    import google.colab
    if SEPA_SOURCE != 'publico':
        from google.colab import drive
        drive.mount('/content/drive')
        print('Google Drive montado en /content/drive')
    else:
        print('Modo público: los datos se descargan automáticamente.')
        print('Resultados en /content/output_canasta (descargables desde el explorador de archivos de Colab).')
except ImportError:
    print('Entorno local detectado')

In [ ]:
# Instalar dependencias
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl', 'tqdm', 'pyarrow', '-q'], check=False)

import zipfile, gzip, io, os, re, shutil, warnings
import requests
from pathlib import Path
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ── Resolver fuente de datos ────────────────────────────────
if SEPA_SOURCE not in ('mi_drive', 'local'):
    raise ValueError(f"SEPA_SOURCE='{SEPA_SOURCE}' no reconocido. Usar 'mi_drive' o 'local'.")

SEPA_DIR   = Path(SEPA_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Directorio de caché de parquets
CACHE_DIR = OUTPUT_DIR / '_cache'
if USE_CACHE:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Salida: {OUTPUT_DIR}')
print(f'Caché:  {"habilitado → " + str(CACHE_DIR) if USE_CACHE else "deshabilitado"}')
print(f'Período: {"autodetectar último mes disponible" if PERIODO is None else PERIODO}')

In [ ]:
# ---- Maestros: local o GitHub ----
GITHUB_RAW = 'https://github.com/santiagoriverti/precios_minoristas_supermercados/raw/main/data/'
GITHUB_URLS = {
    'Maestro de Productos Interno.xlsx': GITHUB_RAW + 'Maestro%20de%20Productos%20Interno.xlsx',
    'maestro_sucursales_completo.xlsx':  GITHUB_RAW + 'maestro_sucursales_completo.xlsx',
    'maestro-provincias.xlsx':           GITHUB_RAW + 'maestro-provincias.xlsx',
}

def resolver_maestro(nombre: str) -> Path:
    """Devuelve el path al maestro: usa directorio local si existe, si no descarga desde GitHub."""
    if MAESTROS_DIR:
        local = Path(MAESTROS_DIR) / nombre
        if local.exists():
            print(f'  Local: {local}')
            return local

    # Fallback: descargar desde GitHub al directorio de salida
    dest = OUTPUT_DIR / nombre
    if dest.exists():
        print(f'  Cache GitHub: {dest}')
        return dest

    print(f'  Descargando desde GitHub: {nombre} ...')
    resp = requests.get(GITHUB_URLS[nombre], timeout=120)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print(f'  Guardado ({len(resp.content)/1024:.0f} KB)')
    return dest

print('Resolviendo maestros...')
MAESTRO_PRODUCTOS_PATH   = resolver_maestro('Maestro de Productos Interno.xlsx')
MAESTRO_SUCURSALES_PATH  = resolver_maestro('maestro_sucursales_completo.xlsx')
MAESTRO_PROVINCIAS_PATH  = resolver_maestro('maestro-provincias.xlsx')
print('Maestros listos')

## 2. Carga de datos SEPA

Formato de los archivos: `MMAAAA_pais_parteN_COMPLETO.csv.gz` dentro del ZIP semestral.
- **Parte 1:** días 1–15 | **Parte 2:** días 16–30
- **Precios:** entero en centavos, `NA` si la sucursal no reportó ese día

In [ ]:
_TMP_DIR = Path('/content/tmp_sepa')
_TMP_DIR.mkdir(exist_ok=True)

def cargar_sepa(zip_path: Path, filename: str) -> pd.DataFrame:
    """
    Lee un .csv.gz desde dentro de un .zip con uso mínimo de RAM:
    - Extrae el .csv.gz a disco en streaming (no carga en RAM)
    - Lee el CSV en chunks de 200k filas
    - Reduce a 8 columnas antes de acumular (precio float32)

    Los precios se devuelven SIN divisor (valores crudos del archivo).
    El factor de conversión centavos→pesos se autodetecta y aplica en la
    celda siguiente, después de consolidar ambas partes.
    """
    print(f'  Leyendo {filename} ...')

    # Extraer .csv.gz a disco en streaming (evita cargarlo completo en RAM)
    tmp_path = _TMP_DIR / filename
    with zipfile.ZipFile(zip_path, 'r') as z:
        with z.open(filename) as src, open(tmp_path, 'wb') as dst:
            shutil.copyfileobj(src, dst, length=4 * 1024 * 1024)

    n_dias  = None
    chunks  = []
    with gzip.open(tmp_path, 'rt', encoding='utf-8') as g:
        for chunk in pd.read_csv(
            g,
            dtype={'id_comercio': 'str', 'id_bandera': 'str',
                   'id_sucursal': 'str',  'id_producto': 'str',
                   'sucursales_provincia': 'str'},
            chunksize=200_000,
            low_memory=False
        ):
            price_cols = [c for c in chunk.columns if c.startswith('precio_')]
            if n_dias is None:
                n_dias = len(price_cols)
            # Valores crudos en float32 — sin divisor
            prices = chunk[price_cols].replace('NA', np.nan).astype('float32')
            chunk  = chunk.drop(columns=price_cols)
            chunk['precio_promedio']  = prices.mean(axis=1).astype('float32')
            chunk['dias_con_precio']  = prices.notna().sum(axis=1).astype('int16')
            chunk['total_dias_parte'] = np.int16(n_dias)
            del prices
            chunks.append(chunk[['id_comercio', 'id_bandera', 'id_sucursal',
                                  'sucursales_provincia', 'id_producto',
                                  'precio_promedio', 'dias_con_precio', 'total_dias_parte']])

    tmp_path.unlink()
    df = pd.concat(chunks, ignore_index=True)
    del chunks; gc.collect()
    print(f'    -> {len(df):,} filas | {df["id_producto"].nunique():,} productos | {n_dias} días')
    return df


def detectar_ultimo_mes(sepa_dir: Path, periodo_forzado: str = None) -> tuple:
    # Escanea todos los ZIPs en sepa_dir y devuelve el último mes disponible.
    # Si periodo_forzado está seteado ('YYYY-MM'), busca ese mes específico.
    #
    # Estructura esperada:
    #   carga/
    #     YYYYX.zip  (X = A ene-jun, B jul-dic)
    #       MMYYYY_pais_parteNCOMPLETO[b].csv.gz
    #
    # Returns:
    #     zip_path (Path): ZIP que contiene el mes
    #     periodo  (str):  'YYYY-MM' del mes detectado
    #     archivos (list): nombres de archivos para ese mes (ordenados)
    patron = re.compile(r'^(\d{2})(\d{4})_pais_parte.*COMPLETO.*\.csv\.gz$')
    meses  = {}   # (anio, mes) -> {'zip': Path, 'archivos': [str]}

    for zip_path in sorted(sepa_dir.glob('*.zip')):
        try:
            with zipfile.ZipFile(zip_path, 'r') as z:
                nombres = z.namelist()
        except Exception as e:
            print(f'  ⚠️ No se pudo leer {zip_path.name}: {e}')
            continue

        for nombre in nombres:
            base = Path(nombre).name
            m    = patron.match(base)
            if not m:
                continue
            mes_num, anio_num = int(m.group(1)), int(m.group(2))
            key = (anio_num, mes_num)
            if key not in meses:
                meses[key] = {'zip': zip_path, 'archivos': []}
            meses[key]['archivos'].append(nombre)

    if not meses:
        raise RuntimeError(
            f'No se encontraron archivos SEPA en ningún ZIP de {sepa_dir}'
        )

    if periodo_forzado:
        anio_f, mes_f = int(periodo_forzado[:4]), int(periodo_forzado[5:7])
        key = (anio_f, mes_f)
        if key not in meses:
            disponibles = [f"{a}-{m:02d}" for a, m in sorted(meses.keys())]
            raise RuntimeError(
                f"Período '{periodo_forzado}' no encontrado en ningún ZIP.\n"
                f"Períodos disponibles: {disponibles}"
            )
    else:
        key = max(meses.keys())   # último (anio, mes) disponible

    anio, mes = key
    info      = meses[key]
    periodo   = f'{anio}-{mes:02d}'
    return info['zip'], periodo, sorted(info['archivos'])


In [ ]:
# ── Autodetectar o forzar el período ────────────────────────────────────
ZIP_PATH, PERIODO, _archivos_mes = detectar_ultimo_mes(SEPA_DIR, PERIODO)

NOMBRES_MES = {
    '01': 'enero',    '02': 'febrero', '03': 'marzo',
    '04': 'abril',    '05': 'mayo',    '06': 'junio',
    '07': 'julio',    '08': 'agosto',  '09': 'septiembre',
    '10': 'octubre',  '11': 'noviembre', '12': 'diciembre',
}
_mes_str = NOMBRES_MES.get(PERIODO[5:7], PERIODO[5:7]).capitalize()
print(f'Cargando SEPA {_mes_str} {PERIODO[:4]}...')
print(f'ZIP:    {ZIP_PATH}  ({ZIP_PATH.stat().st_size / 1024**3:.2f} GB)')
print(f'Archivos del mes ({len(_archivos_mes)}):')
for _f in _archivos_mes:
    print(f'  - {_f}')
print('-' * 55)

_cache_suc = CACHE_DIR / f'df_suc_{PERIODO}.parquet' if USE_CACHE else None

if USE_CACHE and _cache_suc.exists():
    # ── Retomar desde caché ─────────────────────────────────────────────────
    print(f'Cargando desde caché: {_cache_suc}')
    df_suc = pd.read_parquet(_cache_suc)
    FACTOR_PRECIO = 1   # el factor ya fue aplicado al guardar el caché
    print(f'  {len(df_suc):,} filas cargadas desde caché (factor ya aplicado)')

else:
    # ── Cargar todos los archivos del mes detectado ────────────────────
    _partes = [cargar_sepa(ZIP_PATH, archivo) for archivo in _archivos_mes]

    df_abril = pd.concat(_partes, ignore_index=True)
    del _partes; gc.collect()

    # Consolidar por (producto × sucursal)
    df_suc = df_abril.groupby(
        ['id_producto', 'id_bandera', 'id_comercio', 'id_sucursal', 'sucursales_provincia'],
        as_index=False
    ).agg(
        precio_promedio = ('precio_promedio',  'mean'),
        dias_con_precio = ('dias_con_precio',  'sum'),
        total_dias      = ('total_dias_parte', 'sum')
    )
    df_suc['pct_dias'] = df_suc['dias_con_precio'] / df_suc['total_dias']
    del df_abril; gc.collect()

    # ── Autodetección del factor de precios ─────────────────────────────────
    # SEPA semestral pre-2025: precios en centavos (ej: 150000 = $1.500 ARS)
    # SEPA semestral 2025B+:   precios ya en pesos  (ej: 1500   = $1.500 ARS)
    # Umbral: si la mediana supera $10.000, los valores están en centavos
    _mediana_ref = df_suc['precio_promedio'].median()
    FACTOR_PRECIO = 100 if _mediana_ref > 10_000 else 1
    print(f'\nDetección de escala de precios:')
    print(f'  Mediana de precios (raw):  {_mediana_ref:,.1f}')
    if FACTOR_PRECIO == 100:
        print(f'  → FACTOR = 100  (datos en centavos — dividiendo ÷100)')
        df_suc['precio_promedio'] = (df_suc['precio_promedio'] / 100).astype('float32')
    else:
        print(f'  → FACTOR = 1    (datos ya en pesos argentinos ✓)')

    # ── Guardar caché ────────────────────────────────────────────────────────
    if USE_CACHE:
        df_suc.to_parquet(_cache_suc, compression='snappy', index=False)
        print(f'  Caché guardado: {_cache_suc}')

print(f'\nDatos consolidados (producto × sucursal):')
print(f'  Filas:            {len(df_suc):,}')
print(f'  Productos únicos: {df_suc["id_producto"].nunique():,}')
print(f'  Cadenas:          {df_suc["id_bandera"].nunique()}')
print(f'  Provincias:       {df_suc["sucursales_provincia"].nunique()}')
print(f'  Sucursales:       {df_suc["id_sucursal"].nunique():,}')

In [ ]:
# Verificación de escala de precios
# Los 10 productos más reportados muestran precios representativos.
# Esperado: valores en el rango ~$100 – ~$20.000 ARS para productos de supermercado.
# Si los precios parecen 100x incorrectos → cambiar USE_CACHE=False y
# borrar el caché para que se re-detecte el factor.
print(f'=== Verificación de escala de precios (factor detectado: ÷{FACTOR_PRECIO}) ===')
top_obs = (
    df_suc.groupby('id_producto')
    .agg(n_sucursales=('id_sucursal','count'), precio_mediano=('precio_promedio','median'))
    .sort_values('n_sucursales', ascending=False)
    .head(10).reset_index()
)
print(top_obs.to_string(index=False))

## 3. Carga de maestros

In [ ]:
print('Cargando Maestro de Productos...')
df_prod = pd.read_excel(MAESTRO_PRODUCTOS_PATH, dtype={'producto_sepa_id': str})
df_prod['id_producto'] = df_prod['producto_sepa_id'].str.strip()
df_prod = df_prod[df_prod['producto_blacklist'] == 0].copy()

df_prod_uniq = (
    df_prod[['id_producto', 'producto_descripcion', 'producto_marca',
             'rubro', 'categoria', 'subcategoria',
             'producto_cantidad_presentacion', 'producto_unidad_medida_presentac']]
    .drop_duplicates('id_producto')
)
print(f'  Productos únicos (sin blacklist): {len(df_prod_uniq):,}')
print(f'  Rubros disponibles ({df_prod_uniq["rubro"].nunique()}):')
print(df_prod_uniq['rubro'].value_counts().to_string())

In [ ]:
print('Cargando Maestro de Sucursales...')
df_suc_maest = pd.read_excel(
    MAESTRO_SUCURSALES_PATH,
    dtype={'id_comercio': str, 'id_bandera': str, 'id_sucursal': str}
)
df_suc_maest['REGION'] = df_suc_maest['REGION'].str.strip()

print(f'  Total sucursales: {len(df_suc_maest):,}')
print(f'  Cadenas:          {df_suc_maest["id_bandera"].nunique()}')
print(f'  Regiones ({df_suc_maest["REGION"].nunique()}):')
print(df_suc_maest.groupby('REGION')['id_sucursal'].nunique().sort_values(ascending=False).to_string())
print('\nSucursales por cadena:')
print(df_suc_maest.groupby('id_bandera')['id_sucursal'].nunique().sort_values(ascending=False).to_string())

print('\nCargando Maestro de Provincias...')
df_provincias = pd.read_excel(
    MAESTRO_PROVINCIAS_PATH,
    dtype={'sucursales_provincia': str, 'provincia': str}
)
df_provincias['sucursales_provincia'] = df_provincias['sucursales_provincia'].str.strip()
df_provincias['provincia']            = df_provincias['provincia'].str.strip()
print(f'  Provincias mapeadas: {len(df_provincias):,}')
print(df_provincias.sort_values('sucursales_provincia').to_string(index=False))

## 4. Enriquecimiento y exploración

In [ ]:
# ── Diccionario de nombres de cadenas comerciales ──────────────────────────
# En formato semestral, id_bandera (1-6) identifica el GRUPO CORPORATIVO,
# no la cadena comercial. La cadena real se obtiene combinando (id_comercio, id_bandera).
_CADENAS_COMPUESTAS = {
    ('9',  '1'): 'Vea',
    ('9',  '2'): 'Disco',
    ('9',  '3'): 'Jumbo',
    ('10', '1'): 'Carrefour',
    ('10', '2'): 'Carrefour Market',
    ('10', '3'): 'Carrefour Express',
    ('11', '2'): 'ChangoMas',
    ('11', '4'): 'Hiper ChangoMas',
    ('11', '5'): 'Mi ChangoMas',
    ('16', '1'): 'Hipermercado Libertad',
    ('16', '2'): 'Mini Libertad',
}
_CADENAS_SIMPLES = {
    '2':  'La Anónima',
    '3':  'Cadena 3',
    '5':  'Hipermercado Misiones',
    '8':  'Cadena 8 (Córdoba)',
    '12': 'Coto',
    '13': 'Cooperativa Obrera',
    '15': 'DIA',
    '20': 'LAR',
    '21': 'Toledo',
    '23': 'Cadena 23',
    '47': 'Pasamonte',
}

# ─────────────────────────────────────────────────────────────────────────────
# DISEÑO ANTI-OOM
#
# El problema original era mantener df_enr (~50M filas × 20 columnas, ~10 GB)
# vivo desde la celda de enriquecimiento hasta los heatmaps.
#
# Solución: agregar INMEDIATAMENTE a dos estructuras pequeñas:
#
#   df_cov        — producto × cadena × provincia  (~2M filas × 8 col)   COBERTURA
#   df_price_stats — producto                       (~170K filas × 5 col)  PRECIOS
#
# El frame intermedio df_suc_enr (~50M filas × 12 col, ~6 GB) se borra en
# cuanto termina la agregación. La RAM pico baja de ~10 GB a ~600 MB.
# ─────────────────────────────────────────────────────────────────────────────

# ── Paso 1: merge df_suc con geografía de sucursales ─────────────────────────
suc_geo = df_suc_maest[['id_comercio', 'id_bandera', 'id_sucursal',
                          'PROVINCIA', 'REGION']].copy()
df_suc_enr = df_suc.merge(suc_geo, on=['id_comercio', 'id_bandera', 'id_sucursal'], how='left')
del df_suc, suc_geo; gc.collect()

# ── Paso 2: nombre de provincia legible ──────────────────────────────────────
df_suc_enr = df_suc_enr.merge(
    df_provincias[['sucursales_provincia', 'provincia']],
    on='sucursales_provincia', how='left'
)
df_suc_enr['PROVINCIA_NOMBRE'] = (
    df_suc_enr['PROVINCIA'].combine_first(df_suc_enr['provincia'])
    .str.strip()
    .str.replace(r'^Provincia de ', '', regex=True)
    .str.replace('Ciudad Autónoma de Buenos Aires', 'CABA', regex=False)
    .str.title()
    .str.replace('Caba', 'CABA', regex=False)
)
df_suc_enr.drop(columns=['PROVINCIA', 'provincia', 'sucursales_provincia',
                          'dias_con_precio', 'total_dias'], inplace=True, errors='ignore')
df_suc_enr['REGION'] = df_suc_enr['REGION'].str.strip()

# ── Paso 3: nombre de cadena comercial (vectorizado — sin apply fila a fila) ─
_lookup_comp = {f"{k[0]}_{k[1]}": v for k, v in _CADENAS_COMPUESTAS.items()}
_ck = df_suc_enr['id_comercio'] + '_' + df_suc_enr['id_bandera']
df_suc_enr['nombre_cadena'] = _ck.map(_lookup_comp)
_null = df_suc_enr['nombre_cadena'].isna()
df_suc_enr.loc[_null, 'nombre_cadena'] = df_suc_enr.loc[_null, 'id_comercio'].map(_CADENAS_SIMPLES)
_null = df_suc_enr['nombre_cadena'].isna()
df_suc_enr.loc[_null, 'nombre_cadena'] = 'Comercio ' + df_suc_enr.loc[_null, 'id_comercio']
del _ck, _null; gc.collect()

# ── Paso 4: estadísticas de precio a nivel sucursal (antes de agregar) ────────
# Los percentiles requieren la distribución a nivel sucursal.
# Una vez colapsado a producto×cadena×provincia los perdemos.
df_price_stats = (
    df_suc_enr.groupby('id_producto', sort=False)['precio_promedio']
    .agg(precio_promedio='mean', precio_mediano='median')
    .astype('float32')
    .reset_index()
)
_pq = (
    df_suc_enr.groupby('id_producto', sort=False)['precio_promedio']
    .quantile([0.25, 0.75])
    .unstack()
    .rename(columns={0.25: 'precio_p25', 0.75: 'precio_p75'})
    .astype('float32')
    .reset_index()
)
df_price_stats = df_price_stats.merge(_pq, on='id_producto', how='left')
del _pq

# ── Paso 5: agregar a (producto × cadena × provincia) ─────────────────────────
# REDUCCIÓN CRÍTICA: ~50M filas → ~2M filas — libera ~8–15 GB de RAM.
df_cov = (
    df_suc_enr
    .groupby(['id_producto', 'id_bandera', 'nombre_cadena', 'PROVINCIA_NOMBRE', 'REGION'],
             sort=False, dropna=False)
    .agg(
        n_sucursales    = ('id_sucursal',    'count'),
        pct_dias        = ('pct_dias',        'mean'),
        precio_promedio = ('precio_promedio', 'mean'),
    )
    .reset_index()
)
del df_suc_enr; gc.collect()   # ← LIBERACIÓN CRÍTICA: RAM pasa de ~10 GB a ~600 MB
print(f'df_cov: {len(df_cov):,} filas × {df_cov.shape[1]} columnas')
print(f'df_price_stats: {len(df_price_stats):,} filas × {df_price_stats.shape[1]} columnas')

# ── Paso 6: cadenas por producto (desde frame deduplicado — máx 16 filas/prod) ─
_cad_dedup = df_cov[['id_producto', 'nombre_cadena']].drop_duplicates()
_cad_agg = (
    _cad_dedup.groupby('id_producto', sort=False)
    .agg(
        n_cadenas_com     = ('nombre_cadena', 'nunique'),
        cadenas_presentes = ('nombre_cadena',
                             lambda x: ', '.join(sorted(str(v) for v in x if pd.notna(v))))
    )
    .reset_index()
)
del _cad_dedup

# ── Paso 7: añadir metadata de productos a df_cov ─────────────────────────────
# df_cov tiene ~2M filas — la merge con df_prod_uniq (~170K filas) es barata.
df_cov = df_cov.merge(df_prod_uniq, on='id_producto', how='left')
gc.collect()

# ── Diagnóstico ──────────────────────────────────────────────────────────────
print(f'\nMatch maestro sucursales: {df_cov["REGION"].notna().mean()*100:.1f}% de filas')
print(f'Match maestro productos:  {df_cov["rubro"].notna().mean()*100:.1f}% de filas')
print(f'Match maestro provincias: {df_cov["PROVINCIA_NOMBRE"].notna().mean()*100:.1f}% de filas')
print(f'Productos sin clasificar: {df_cov[df_cov["rubro"].isna()]["id_producto"].nunique():,}')

print('\nCadenas comerciales identificadas:')
print(df_cov.groupby('nombre_cadena')['id_producto'].nunique()
      .sort_values(ascending=False).rename('productos').to_string())

print('\nProvincias activas (normalizadas):')
print(df_cov.groupby('PROVINCIA_NOMBRE')['id_producto'].nunique()
      .sort_values(ascending=False).rename('productos').to_string())

In [ ]:
print('=== Distribución por cadena comercial — Abril 2026 ===')
# df_cov tiene una fila por (producto × cadena × provincia).
# Para totales por cadena: sumamos n_sucursales y contamos productos únicos.
print(df_cov.groupby('nombre_cadena').agg(
    sucursales_activas   = ('n_sucursales',     'sum'),
    productos_reportados = ('id_producto',      'nunique'),
    precio_mediano       = ('precio_promedio',  'median'),
).sort_values('productos_reportados', ascending=False).to_string())

print('\n=== Distribución por región geográfica ===')
print(df_cov.groupby('REGION').agg(
    sucursales_activas   = ('n_sucursales', 'sum'),
    productos_reportados = ('id_producto',  'nunique'),
).sort_values('sucursales_activas', ascending=False).to_string())

In [ ]:
print('=== Top 20 productos más reportados ===')
top_prod = (
    df_cov.groupby('id_producto', sort=False)
    .agg(
        n_sucursales         = ('n_sucursales',          'sum'),
        n_cadenas            = ('id_bandera',             'nunique'),
        n_regiones           = ('REGION',                 'nunique'),
        precio_med           = ('precio_promedio',        'median'),
        producto_descripcion = ('producto_descripcion',   'first'),
        producto_marca       = ('producto_marca',          'first'),
        rubro                = ('rubro',                   'first'),
        categoria            = ('categoria',               'first'),
    )
    .sort_values('n_sucursales', ascending=False).head(20).reset_index()
)
print(top_prod[['id_producto', 'producto_descripcion', 'producto_marca',
                'rubro', 'n_cadenas', 'n_regiones', 'n_sucursales', 'precio_med']].to_string(index=False))

## 5. Análisis de cobertura

Para cada producto:
- **n_cadenas / n_provincias / n_sucursales**: cobertura por cadena y geográfica (provincial)
- **pct_dias_promedio**: % de días de abril con precio reportado
- **score_cobertura**: 50% cobertura cadenas + 50% cobertura provincias, ponderado por continuidad

Los umbrales `MIN_CADENAS` y `MIN_PROVINCIAS` son **dinámicos**: se calculan a partir de los datos reales (total de cadenas y provincias activas en el período). Solo pasan los productos presentes en **todas** las cadenas y **todas** las provincias.

In [ ]:
total_cadenas    = df_cov['id_bandera'].nunique()
total_provincias = df_cov['PROVINCIA_NOMBRE'].nunique()   # nunique() omite NaN

MIN_CADENAS    = total_cadenas
MIN_PROVINCIAS = total_provincias

print(f'Grupos corporativos activos: {total_cadenas}   → MIN_CADENAS    = {MIN_CADENAS}')
print(f'Cadenas comerciales reales:  {df_cov["nombre_cadena"].nunique()}  (informativo)')
print(f'Provincias activas:          {total_provincias} → MIN_PROVINCIAS = {MIN_PROVINCIAS}')

# ── Agregación de cobertura a nivel producto ──────────────────────────────────
# Solo aggregators nativos de pandas (C interno) — sin Python lambdas en
# conjuntos de millones de filas. Precios y cadenas se unen en pasos separados.
df_cob = (
    df_cov.groupby('id_producto', sort=False)
    .agg(
        n_cadenas         = ('id_bandera',                        'nunique'),
        n_provincias      = ('PROVINCIA_NOMBRE',                  'nunique'),
        n_regiones        = ('REGION',                            'nunique'),
        n_sucursales      = ('n_sucursales',                      'sum'),
        pct_dias_promedio = ('pct_dias',                           'mean'),
        rubro             = ('rubro',                             'first'),
        categoria         = ('categoria',                         'first'),
        subcategoria      = ('subcategoria',                      'first'),
        descripcion       = ('producto_descripcion',              'first'),
        marca             = ('producto_marca',                    'first'),
        presentacion      = ('producto_cantidad_presentacion',    'first'),
        unidad            = ('producto_unidad_medida_presentac',  'first'),
    )
    .reset_index()
)

# Unir estadísticas de precio (calculadas antes de agregar — percentiles correctos)
df_cob = df_cob.merge(df_price_stats, on='id_producto', how='left')

# Unir cadenas por producto
df_cob = df_cob.merge(_cad_agg, on='id_producto', how='left')

# ── Score de cobertura ────────────────────────────────────────────────────────
df_cob['pct_cadenas']     = df_cob['n_cadenas']    / total_cadenas
df_cob['pct_provincias']  = df_cob['n_provincias'] / total_provincias
df_cob['score_cobertura'] = (
    (df_cob['pct_cadenas'] * 0.5 + df_cob['pct_provincias'] * 0.5)
    * df_cob['pct_dias_promedio']
)

print(f'\nProductos con al menos 1 observación: {len(df_cob):,}')
print(df_cob[['n_cadenas', 'n_cadenas_com', 'n_provincias', 'n_sucursales', 'pct_dias_promedio']].describe().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribución de Cobertura — SEPA Abril 2026', fontsize=14, fontweight='bold')

for ax, col, bins, color, umbral, label in [
    (axes[0,0], 'n_cadenas',        range(0, total_cadenas+2),    'steelblue',   MIN_CADENAS,    'Cadenas'),
    (axes[0,1], 'n_provincias',     range(0, total_provincias+2), 'seagreen',    MIN_PROVINCIAS, 'Provincias'),
    (axes[1,0], 'n_sucursales',      40,                          'darkorange',  MIN_SUCURSALES, 'Sucursales'),
    (axes[1,1], 'pct_dias_promedio', 25,                          'mediumpurple',MIN_PCT_DIAS,   '% días'),
]:
    data = df_cob[col].clip(upper=600) if col == 'n_sucursales' else df_cob[col]
    kwargs = {'align': 'left'} if isinstance(bins, range) else {}
    ax.hist(data, bins=bins, color=color, edgecolor='white', **kwargs)
    vline = umbral - 0.5 if col in ('n_cadenas', 'n_provincias') else umbral
    ax.axvline(vline, color='crimson', linestyle='--', linewidth=1.5, label=f'Umbral: {umbral}')
    ax.set_title(f'N° de {label.lower()} por producto')
    ax.set_xlabel(label); ax.legend()
    if col == 'pct_dias_promedio':
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    elif isinstance(bins, range):
        ax.xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_distribucion_cobertura.png', dpi=150, bbox_inches='tight')
plt.show()

n_todos = (
    (df_cob['n_cadenas']         >= MIN_CADENAS)    &
    (df_cob['n_provincias']      >= MIN_PROVINCIAS) &
    (df_cob['n_sucursales']      >= MIN_SUCURSALES) &
    (df_cob['pct_dias_promedio'] >= MIN_PCT_DIAS)
).sum()
print(f'Productos que superan TODOS los umbrales: {n_todos:,} / {len(df_cob):,}')

In [ ]:
candidatos = df_cob[
    (df_cob['n_cadenas']         >= MIN_CADENAS)    &
    (df_cob['n_provincias']      >= MIN_PROVINCIAS) &
    (df_cob['n_sucursales']      >= MIN_SUCURSALES) &
    (df_cob['pct_dias_promedio'] >= MIN_PCT_DIAS)
].copy()
cand_con_maestro = candidatos[candidatos['rubro'].notna()].copy()

print(f'Productos candidatos (todos los filtros): {len(candidatos):,}')
print(f'  Con clasificación en maestro:           {len(cand_con_maestro):,}')
print(f'  Sin clasificar:                         {len(candidatos) - len(cand_con_maestro):,}')
print('\nCandidatos por rubro:')
print(cand_con_maestro['rubro'].value_counts().to_string())

In [ ]:
# Heatmaps: top 40 candidatos × cadenas y × provincias.
# df_cov tiene pct_dias a nivel (producto × cadena × provincia) — exactamente
# lo que necesitan los heatmaps. No se necesita df_enr.
top_ids = cand_con_maestro.sort_values('score_cobertura', ascending=False).head(40)['id_producto'].tolist()
df_heat = (
    df_cov[df_cov['id_producto'].isin(top_ids)]
    .merge(candidatos[['id_producto', 'descripcion']], on='id_producto', how='left')
)
df_heat['label'] = df_heat['descripcion'].str[:45].fillna(df_heat['id_producto'])

for pivot_col, cmap, fname, title_suffix in [
    ('nombre_cadena',    'YlGnBu', '02_heatmap_cadenas.png',    '× cadena comercial'),
    ('PROVINCIA_NOMBRE', 'RdYlGn', '03_heatmap_provincias.png', '× provincia'),
]:
    pivot = (
        df_heat
        .dropna(subset=[pivot_col])
        .groupby(['label', pivot_col])['pct_dias'].mean()
        .unstack(fill_value=0)
    )
    fig, ax = plt.subplots(figsize=(10, 14))
    sns.heatmap(pivot, cmap=cmap, linewidths=0.4, linecolor='white', vmin=0, vmax=1,
                cbar_kws={'label': '% días con precio', 'shrink': 0.6}, ax=ax)
    ax.set_title(f'Cobertura — Top 40 candidatos {title_suffix}', fontsize=13, pad=12)
    ax.tick_params(axis='y', labelsize=8)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / fname, dpi=150, bbox_inches='tight')
    plt.show()

# Liberar df_cov — ya no se necesita después de los heatmaps
del df_heat, df_cov; gc.collect()

## 6. Construcción de la canasta representativa

Estructura basada en la **Canasta Básica Alimentaria (CBA)** del INDEC para familia tipo de 4 integrantes, adaptada a los rubros del SEPA.

Para cada grupo se define:
- **rubros**: rubros del maestro de productos incluidos
- **kw**: palabras clave en la columna `categoria` que **incluyen** un producto
- **excluir_kw**: palabras clave en `categoria` que **excluyen** un producto (evita falsos positivos)

Dentro de cada grupo se seleccionan los productos con mayor `score_cobertura`.

In [ ]:
# ─── Definición de grupos de la canasta ───────────────────────────────────────
# rubros        : rubros del maestro de productos que aplican al grupo
# kw            : palabras clave que deben aparecer en la columna 'categoria'
# excluir_kw    : palabras clave en 'categoria' que EXCLUYEN el producto del grupo
#                 (evita falsos positivos por substring, ej. 'crema' en 'Mayonesa con Crema')
# excluir_subcat: valores exactos de 'subcategoria' que excluyen el producto
#                 (más preciso que excluir_kw cuando una misma categoria agrupa tipos
#                  de producto distintos, ej. 'Fiambrería' incluye fiambres Y quesos)
# max           : cantidad máxima de productos en el grupo
# ──────────────────────────────────────────────────────────────────────────────
GRUPOS_CANASTA = {
    'Cereales y derivados': {
        'rubros':         ['Almacén'],
        'kw':             ['arroz', 'pasta', 'fideo', 'harina', 'galletita', 'cereal', 'pan'],
        'excluir_kw':     None,
        'excluir_subcat': None,
        'max': 8,
    },
    'Lácteos': {
        'rubros':         ['Frescos'],
        # FIX BUG-6: el maestro SEPA almacena categoria='Lácteos' (string literal).
        # Las kw anteriores ['leche','yogur','queso',...] buscaban como substring en
        # esa columna y nunca coincidían → grupo vacío. Fix: usar el valor real.
        'kw':             ['lácteos', 'lacteos'],
        'excluir_kw':     None,
        'excluir_subcat': None,
        'max': 8,
    },
    'Aceites y grasas': {
        'rubros':         ['Almacén'],
        'kw':             ['aceite', 'manteca', 'margarina'],
        'excluir_kw':     None,
        'excluir_subcat': None,
        'max': 4,
    },
    'Azúcar, dulces y conservas': {
        'rubros':         ['Almacén'],
        'kw':             ['azúcar', 'azucar', 'mermelada', 'dulce', 'tomate', 'conserva', 'legumbre'],
        'excluir_kw':     None,
        # FIX BUG-7: categoria='Conservas' agrupa frutas en almíbar Y carnes enlatadas.
        # excluir_kw no puede distinguirlos porque ambos tienen el mismo valor de
        # categoria. La columna subcategoria sí los separa → excluir por subcategoria.
        'excluir_subcat': ['Patés y Picadillos', 'Conservas de Pescado'],
        'max': 6,
    },
    'Carnes y fiambres': {
        'rubros':         ['Frescos', 'Almacén', 'Congelados'],
        'kw':             ['fiambre', 'embutido', 'carne', 'carnicería', 'carniceria', 'salchicha', 'pollo', 'atún', 'atun'],
        'excluir_kw':     ['dulce', 'mermelada', 'postre', 'conserva', 'galletita'],
        # FIX BUG-8 + BUG-11: categoria='Fiambrería' mezcla fiambres con quesos (69 Quesos Untables,
        # 24 Semiduros, etc.). El excluir_kw anterior ['queso untable'] buscaba ese string
        # en la columna categoria, pero todos tienen categoria='Fiambrería' → inefectivo.
        # Fix: excluir directamente por subcategoria.
        'excluir_subcat': [
            'Quesos Untables', 'Quesos Semiduros', 'Quesos Rallados',
            'Quesos Blandos',  'Quesos Duros',     'Quesos Especiales',
            'Dulces',
        ],
        'max': 6,
    },
    'Huevos': {
        'rubros':         ['Frescos', 'Almacén'],
        'kw':             ['huevo'],
        'excluir_kw':     None,
        'excluir_subcat': None,
        'max': 2,
    },
    'Condimentos y aderezos': {
        'rubros':         ['Almacén'],
        'kw':             ['salsa', 'condimento', 'vinagre', 'mayonesa', 'mostaza', 'ketchup', 'aderezo'],
        'excluir_kw':     None,
        'excluir_subcat': None,
        'max': 5,
    },
    'Bebidas no alcohólicas': {
        # FIX BUG-9: yerba mate, té y café están en rubro='Almacén', categoria='Infusiones'.
        # Antes solo se buscaba en rubro='Bebidas' → nunca aparecían.
        # Fix: agregar 'Almacén' a rubros; 'infusion' matchea 'Infusiones' como substring.
        'rubros':         ['Bebidas', 'Almacén'],
        'kw':             ['agua', 'gaseosa', 'jugo', 'saborizada', 'infusion', 'bebida herbal'],
        'excluir_kw':     ['vino', 'espumante', 'cerveza', 'sidra', 'fernet',
                           'aperitivo', 'licor', 'whisky', 'ron', 'vodka', 'gin'],
        'excluir_subcat': None,
        'max': 8,   # aumentado: compiten jugos + infusiones (yerba/té/café) + aguas + gaseosas
    },
    'Bebidas alcohólicas': {
        'rubros':         ['Bebidas'],
        'kw':             ['vino', 'espumante', 'cerveza', 'sidra', 'fernet',
                           'aperitivo', 'licor', 'whisky', 'ron', 'vodka', 'gin'],
        'excluir_kw':     None,
        'excluir_subcat': None,
        'max': 6,   # aumentado: da lugar a cerveza además de vinos/espumantes
    },
    'Limpieza del hogar': {
        'rubros':         ['Limpieza'],
        'kw':             None,
        'excluir_kw':     None,
        # FIX BUG-12: categoria='Accesorios de Limpieza' incluye implementos fisicos
        # (cabos metalicos, escobas, plumeros) junto a productos de limpieza reales.
        # No corresponden a una canasta de precios. Fix: excluir por subcategoria.
        'excluir_subcat': ['Palas y Cabos', 'Escobas y Escobillones', 'Plumeros y Limpiavidrios'],
        'max': 7,
    },
    'Higiene y cuidado personal': {
        'rubros':         ['Perfumería'],
        'kw':             None,
        'excluir_kw':     None,
        # FIX BUG-13: subcategoria='Coloración' (tintura de cabello) y 'Fijación'
        # (protector térmico, spray fijador) tienen alta cobertura pero son beauty/styling,
        # no higiene básica. Excluirlos deja pasar desodorantes, que sí corresponden.
        # IMPORTANTE: los valores deben tener tilde — el maestro los almacena con acento
        # y .isin() hace comparación exacta (case y acento sensitivo).
        'excluir_subcat': ['Coloración', 'Fijación'],
        'max': 6,
    },
}


def seleccionar_grupo(df, rubros, keywords, excluir_kw, max_n, excluir_subcat=None):
    """
    Selecciona productos para un grupo de la canasta.

    Pasos:
      1. Filtra por rubro.
      2. Excluye filas cuya 'categoria' contenga alguna palabra de excluir_kw.
      3. Excluye filas cuya 'subcategoria' esté en excluir_subcat (más preciso que
         excluir_kw cuando una misma categoria agrupa tipos de producto distintos).
      4. Incluye solo filas cuya 'categoria' contenga alguna palabra de keywords.
      5. Ordena por score_cobertura y devuelve los top max_n.

    Sin fallback: si el filtro de keywords deja 0 resultados, el grupo queda vacío.
    Es preferible un grupo vacío a incluir productos incorrectos.
    """
    subset = df[df['rubro'].isin(rubros)].copy()

    if excluir_kw and len(subset) > 0:
        excl_mask = subset['categoria'].str.contains(
            '|'.join(excluir_kw), case=False, na=False
        )
        subset = subset[~excl_mask]

    if excluir_subcat and len(subset) > 0:
        subset = subset[~subset['subcategoria'].isin(excluir_subcat)]

    if keywords and len(subset) > 0:
        incl_mask = subset['categoria'].str.contains(
            '|'.join(keywords), case=False, na=False
        )
        subset = subset[incl_mask]  # sin fallback

    return subset.sort_values('score_cobertura', ascending=False).head(max_n)


partes = []
for grupo, cfg in GRUPOS_CANASTA.items():
    sel = seleccionar_grupo(
        cand_con_maestro,
        cfg['rubros'],
        cfg['kw'],
        cfg['excluir_kw'],
        cfg['max'],
        excluir_subcat=cfg.get('excluir_subcat'),
    ).copy()
    sel['grupo_canasta'] = grupo
    partes.append(sel)
    print(f'{grupo}: {len(sel)} productos')

df_canasta = pd.concat(partes, ignore_index=True).drop_duplicates(subset='id_producto', keep='first')
print(f'\nTotal en la canasta: {len(df_canasta)}')


In [ ]:
cols_show = ['descripcion','marca','presentacion','unidad',
             'n_cadenas','n_provincias','n_sucursales','pct_dias_promedio','precio_mediano']

print('=' * 110)
print('CANASTA REPRESENTATIVA — FAMILIA TIPO 4 INTEGRANTES — ABRIL 2026')
print('=' * 110)

for grupo in GRUPOS_CANASTA:
    gdf = df_canasta[df_canasta['grupo_canasta'] == grupo]
    if len(gdf) == 0:
        print(f'\n[{grupo}] — sin productos que cumplan los umbrales')
        continue
    print(f'\n{"─"*110}\n  {grupo.upper()}  ({len(gdf)} productos)\n{"─"*110}')
    print(gdf[cols_show].sort_values('n_cadenas', ascending=False).to_string(index=False))

In [ ]:
resumen = (
    df_canasta.groupby('grupo_canasta')
    .agg(n_productos=('id_producto','count'), cob_cadenas=('n_cadenas','mean'),
         cob_provincias=('n_provincias','mean'),  precio_mediano=('precio_mediano','median'))
    .reset_index().sort_values('cob_cadenas', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Resumen de la Canasta — Abril 2026', fontsize=13, fontweight='bold')

ax = axes[0]
x, w = range(len(resumen)), 0.35
ax.barh([i+w/2 for i in x], resumen['cob_cadenas'],    w, label='Cadenas prom.',    color='steelblue')
ax.barh([i-w/2 for i in x], resumen['cob_provincias'], w, label='Provincias prom.', color='seagreen')
ax.set_yticks(list(x)); ax.set_yticklabels(resumen['grupo_canasta'], fontsize=9)
ax.set_title('Cobertura promedio por grupo'); ax.legend()

ax = axes[1]
rs = resumen.sort_values('precio_mediano')
ax.barh(rs['grupo_canasta'], rs['precio_mediano'],
        color=plt.cm.RdYlGn(rs['precio_mediano'] / rs['precio_mediano'].max()))
ax.set_title('Precio mediano por grupo (pesos)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_resumen_canasta.png', dpi=150, bbox_inches='tight')
plt.show()

# Box plot de dispersión de precios
orden = df_canasta.groupby('grupo_canasta')['precio_mediano'].median().sort_values(ascending=False).index.tolist()
fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(
    [df_canasta[df_canasta['grupo_canasta']==g]['precio_mediano'].dropna().values for g in orden],
    vert=False, patch_artist=True, medianprops=dict(color='black', linewidth=2)
)
for patch, c in zip(bp['boxes'], plt.cm.tab20.colors):
    patch.set_facecolor(c); patch.set_alpha(0.75)
ax.set_yticks(range(1, len(orden)+1)); ax.set_yticklabels(orden, fontsize=9)
ax.set_title('Dispersión de precios por grupo — Abril 2026', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_dispersion_precios.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Exportación de resultados

Se genera un único archivo Excel con dos hojas:

| Hoja | Contenido |
|------|-----------|
| **Canasta** | ~60 productos sugeridos, organizados por grupo, con cobertura y precios |
| **Candidatos** | Todos los productos que superan los umbrales de cobertura, para que el economista seleccione su propia canasta |

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ─── Agregar columna de período ────────────────────────────────────────────────
# Permite que el notebook que consuma este Excel sepa a qué período corresponde
# sin depender del nombre del archivo.
df_canasta['periodo']    = PERIODO
cand_con_maestro         = cand_con_maestro.copy()
cand_con_maestro['periodo'] = PERIODO

# ─── Columnas del Excel ────────────────────────────────────────────────────────
# Diseñadas para ser la fuente de alimentación de notebooks posteriores.
# n_cadenas     = grupos corporativos (base del score y los umbrales)
# n_cadenas_com = cadenas comerciales reales (informativo)
# cadenas_presentes = lista de cadenas que reportaron el producto (texto legible)
COLS_CANASTA = [
    'periodo', 'grupo_canasta',
    'id_producto', 'descripcion', 'marca', 'presentacion', 'unidad',
    'rubro', 'categoria',
    'n_cadenas', 'n_cadenas_com', 'n_provincias', 'n_sucursales',
    'pct_dias_promedio',
    'precio_mediano', 'precio_p25', 'precio_p75',
    'score_cobertura',
    'cadenas_presentes',
]
COLS_CANDIDATOS = [
    'periodo',
    'id_producto', 'descripcion', 'marca', 'presentacion', 'unidad',
    'rubro', 'categoria', 'subcategoria',
    'n_cadenas', 'n_cadenas_com', 'n_provincias', 'n_sucursales',
    'pct_dias_promedio',
    'precio_mediano', 'precio_p25', 'precio_p75',
    'score_cobertura',
    'cadenas_presentes',
]

canasta_export    = df_canasta[COLS_CANASTA].sort_values(
    ['grupo_canasta', 'score_cobertura'], ascending=[True, False]
)
candidatos_export = cand_con_maestro[COLS_CANDIDATOS].sort_values(
    'score_cobertura', ascending=False
)
# ─── FIX BUG-10: preservar id_producto como texto (EANs con leading zeros) ────
# int64 pierde los ceros iniciales de EANs cortos (ej. 78933354 → debería ser
# 0000078933354). str.zfill(13) restaura el formato EAN-13 completo.
# Sin esto, joins con otras fuentes que lean el EAN como string fallarían.
canasta_export    = canasta_export.copy()
candidatos_export = candidatos_export.copy()
canasta_export['id_producto']    = canasta_export['id_producto'].astype(str).str.zfill(13)
candidatos_export['id_producto'] = candidatos_export['id_producto'].astype(str).str.zfill(13)

# ─── Hoja Selección: universo amplio con umbrales permisivos ────────────────
# Umbrales más bajos que la Canasta automática para darle al economista
# un pool más amplio de opciones. El score_cobertura en la hoja permite
# que aplique su propio criterio de calidad al armar la canasta.
#   Canasta/Candidatos : ≥5 cadenas, ≥24 provincias, ≥50 sucursales
#   Selección           : ≥3 cadenas, ≥18 provincias, ≥30 sucursales
_sel_wide = df_cob[
    (df_cob['n_cadenas']         >= MIN_CADENAS_SEL)    &
    (df_cob['n_provincias']      >= MIN_PROVINCIAS_SEL) &
    (df_cob['n_sucursales']      >= MIN_SUCURSALES_SEL) &
    (df_cob['pct_dias_promedio'] >= MIN_PCT_DIAS)        &
    (df_cob['rubro'].notna())
].copy()
_sel_wide['periodo']            = PERIODO
_CANT_COLS = ['cantidad_01','cantidad_02','cantidad_03','cantidad_04','cantidad_05','cantidad_06']
_COLS_SEL  = ['periodo'] + _CANT_COLS + [c for c in COLS_CANDIDATOS if c != 'periodo']
seleccion_export                = _sel_wide[COLS_CANDIDATOS].copy()
seleccion_export['id_producto'] = seleccion_export['id_producto'].astype(str).str.zfill(13)
for _cc in _CANT_COLS: seleccion_export[_cc] = pd.NA
seleccion_export                = (
    seleccion_export[_COLS_SEL]
    .sort_values(['rubro', 'categoria', 'score_cobertura'], ascending=[True, True, False])
)
del _sel_wide

# ─── Hoja Productos únicos: todos los productos con maestro (sin umbrales) ────
# Igual que Selección pero sin filtros de n_cadenas / n_provincias / n_sucursales.
# Permite al economista explorar el universo completo de productos disponibles.
_prod_unicos = df_cob[df_cob['rubro'].notna()].copy()
_prod_unicos['periodo'] = PERIODO
productos_unicos_export = _prod_unicos[COLS_CANDIDATOS].copy()
productos_unicos_export['id_producto'] = productos_unicos_export['id_producto'].astype(str).str.zfill(13)
for _cc in _CANT_COLS: productos_unicos_export[_cc] = pd.NA
productos_unicos_export = (
    productos_unicos_export[_COLS_SEL]
    .sort_values(['rubro', 'categoria', 'score_cobertura'], ascending=[True, True, False])
)
del _prod_unicos


# ─── Paleta de colores por grupo ──────────────────────────────────────────────
_GRUPO_COLOR = {
    'Cereales y derivados':       'FFF2CC',
    'Lácteos':                    'DDEEFF',
    'Aceites y grasas':           'FFE8CC',
    'Azúcar, dulces y conservas': 'FFD6D6',
    'Carnes y fiambres':          'FCE5CD',
    'Huevos':                     'FFFACD',
    'Condimentos y aderezos':     'E8F5E9',
    'Bebidas no alcohólicas':     'E3F2FD',
    'Bebidas alcohólicas':        'F3E5F5',
    'Limpieza del hogar':         'E0F2F1',
    'Higiene y cuidado personal': 'FCE4EC',
}

# ─── Estilos base ─────────────────────────────────────────────────────────────
_HDR_FILL  = PatternFill(start_color='1F4E79', end_color='1F4E79', fill_type='solid')
_HDR_FONT  = Font(bold=True, color='FFFFFF', size=10)
_HDR_ALIGN = Alignment(horizontal='center', wrap_text=True, vertical='center')
_PCT_COLS   = {'pct_dias_promedio'}
_SCORE_COLS = {'score_cobertura'}
_PRICE_COLS = {'precio_mediano', 'precio_p25', 'precio_p75'}

_COL_W_CANASTA = {
    'periodo': 10, 'grupo_canasta': 26,
    'id_producto': 16, 'descripcion': 42, 'marca': 22,
    'presentacion': 11, 'unidad': 8, 'rubro': 14, 'categoria': 24,
    'n_cadenas': 11, 'n_cadenas_com': 14, 'n_provincias': 13, 'n_sucursales': 13,
    'pct_dias_promedio': 13,
    'precio_mediano': 14, 'precio_p25': 12, 'precio_p75': 12,
    'score_cobertura': 13,
    'cadenas_presentes': 60,
}
_COL_W_CAND = {
    'periodo': 10,
    'id_producto': 16, 'descripcion': 42, 'marca': 22,
    'presentacion': 11, 'unidad': 8, 'rubro': 14,
    'categoria': 24, 'subcategoria': 24,
    'n_cadenas': 11, 'n_cadenas_com': 14, 'n_provincias': 13, 'n_sucursales': 13,
    'pct_dias_promedio': 13,
    'precio_mediano': 14, 'precio_p25': 12, 'precio_p75': 12,
    'score_cobertura': 13,
    'cadenas_presentes': 60,
}
_COL_W_CAND_SEL = {**_COL_W_CAND, **{c: 10 for c in _CANT_COLS}}


def _format_header(ws, n_cols):
    ws.row_dimensions[1].height = 34
    ws.freeze_panes = 'A2'
    for i in range(1, n_cols + 1):
        cell = ws.cell(row=1, column=i)
        cell.fill      = _HDR_FILL
        cell.font      = _HDR_FONT
        cell.alignment = _HDR_ALIGN


def _set_col_widths(ws, cols, widths_map):
    for i, col in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(i)].width = widths_map.get(col, 12)


def _apply_numeric_formats_by_column(ws, cols):
    """Aplica formatos numéricos columna a columna (eficiente para muchas filas)."""
    for i, col in enumerate(cols, 1):
        if col in _PCT_COLS:
            fmt = '0.0%'
        elif col in _SCORE_COLS:
            fmt = '0.000'
        elif col in _PRICE_COLS:
            fmt = '#,##0.00'
        else:
            continue
        for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=i, max_col=i):
            for cell in row:
                cell.number_format = fmt


# ─── Escribir Excel ────────────────────────────────────────────────────────────
out_excel = OUTPUT_DIR / f'canasta_representativa_{PERIODO}.xlsx'

with pd.ExcelWriter(out_excel, engine='openpyxl') as writer:
    canasta_export.to_excel(writer, sheet_name='Canasta', index=False)
    candidatos_export.to_excel(writer, sheet_name='Candidatos', index=False)
    seleccion_export.to_excel(writer, sheet_name='Selección', index=False)
    productos_unicos_export.to_excel(writer, sheet_name='Productos unicos', index=False)

    # ── Hoja Canasta (pocas filas → formato completo con colores por grupo) ────
    ws_c = writer.sheets['Canasta']
    _format_header(ws_c, len(COLS_CANASTA))
    _set_col_widths(ws_c, COLS_CANASTA, _COL_W_CANASTA)

    _grupo_idx = COLS_CANASTA.index('grupo_canasta')
    for row in ws_c.iter_rows(min_row=2, max_row=ws_c.max_row):
        grupo = row[_grupo_idx].value
        color = _GRUPO_COLOR.get(grupo, 'FFFFFF')
        fill  = PatternFill(start_color=color, end_color=color, fill_type='solid')
        for cell in row:
            cell.fill = fill
            col_name  = COLS_CANASTA[cell.column - 1]
            if col_name in _PCT_COLS:
                cell.number_format = '0.0%'
            elif col_name in _SCORE_COLS:
                cell.number_format = '0.000'
            elif col_name in _PRICE_COLS:
                cell.number_format = '#,##0.00'

    # ── Hoja Candidatos (miles de filas → solo header + anchos + formatos num.) ─
    ws_cand = writer.sheets['Candidatos']
    _format_header(ws_cand, len(COLS_CANDIDATOS))
    _set_col_widths(ws_cand, COLS_CANDIDATOS, _COL_W_CAND)
    _apply_numeric_formats_by_column(ws_cand, COLS_CANDIDATOS)

    # ── Hoja Selección (fuente del próximo notebook) ──────────────────────────
    ws_sel = writer.sheets['Selección']
    _format_header(ws_sel, len(_COLS_SEL))
    _set_col_widths(ws_sel, _COLS_SEL, _COL_W_CAND_SEL)
    _apply_numeric_formats_by_column(ws_sel, _COLS_SEL)
    # Resaltar columnas cantidad_01..cantidad_06 en amarillo
    _CANT_FILL      = PatternFill(start_color='FFF9C4', end_color='FFF9C4', fill_type='solid')
    _cant_col_idxs  = [_COLS_SEL.index(c) + 1 for c in _CANT_COLS]
    _cant_col_first = _cant_col_idxs[0]
    _cant_col_last  = _cant_col_idxs[-1]
    for _ci in _cant_col_idxs:
        ws_sel.cell(1, _ci).fill = PatternFill(start_color='F9A825', end_color='F9A825', fill_type='solid')
    for row in ws_sel.iter_rows(min_row=2, max_row=ws_sel.max_row,
                                min_col=_cant_col_first, max_col=_cant_col_last):
        for cell in row:
            cell.fill          = _CANT_FILL
            cell.number_format = '#,##0.##'
    ws_sel.auto_filter.ref = ws_sel.dimensions

    # ── Hoja Productos únicos (todos los productos con maestro, sin umbrales) ──
    ws_pu = writer.sheets['Productos unicos']
    _format_header(ws_pu, len(_COLS_SEL))
    _set_col_widths(ws_pu, _COLS_SEL, _COL_W_CAND_SEL)
    _apply_numeric_formats_by_column(ws_pu, _COLS_SEL)
    for _ci in _cant_col_idxs:
        ws_pu.cell(1, _ci).fill = PatternFill(start_color='F9A825', end_color='F9A825', fill_type='solid')
    for row in ws_pu.iter_rows(min_row=2, max_row=ws_pu.max_row,
                               min_col=_cant_col_first, max_col=_cant_col_last):
        for cell in row:
            cell.fill          = _CANT_FILL
            cell.number_format = '#,##0.##'
    ws_pu.auto_filter.ref = ws_pu.dimensions

print(f'Excel exportado: {out_excel}')
print()
print('─' * 65)
print(f'CANASTA SUGERIDA — FAMILIA TIPO 4 INTEGRANTES — {PERIODO}')
print('─' * 65)
print(canasta_export.groupby('grupo_canasta').agg(
    productos        = ('id_producto',    'count'),
    cadenas_prom     = ('n_cadenas',      'mean'),
    cadenas_com_prom = ('n_cadenas_com',  'mean'),
    provincias_prom  = ('n_provincias',   'mean'),
    precio_mediano   = ('precio_mediano', 'median'),
).round(1).to_string())
print()
print(f'Total canasta:                 {len(canasta_export):>5} productos  →  hoja "Canasta"')
print(f'Candidatos para el economista: {len(candidatos_export):>5} productos  →  hoja "Candidatos"')
print(f'Selección del economista:      {len(seleccion_export):>5} productos  →  hoja "Selección"')
print(f'Productos únicos (con maestro): {len(productos_unicos_export):>5} productos  →  hoja "Productos unicos"')
print(f'  (umbrales amplios: ≥{MIN_CADENAS_SEL} cadenas, ≥{MIN_PROVINCIAS_SEL} provincias, ≥{MIN_SUCURSALES_SEL} sucursales — completar columnas cantidad_01...cantidad_06)')